In [11]:
# ==========================================
# CELDA 1: Configuración y Carga de Infraestructura
# ==========================================
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from ortools.sat.python import cp_model
import warnings
warnings.filterwarnings('ignore')

user = "Joasro"
password = "Akriila123." 
host = "localhost"
db = "dss_academico_unah"

engine = create_engine(f"mysql+mysqlconnector://{user}:{password}@{host}/{db}")
print("Cargando logística desde la BD...")

# 1. Extracción de Docentes
df_docentes = pd.read_sql("""
    SELECT d.ID_Docente, d.Nombre, d.Tipo_Docente, d.Hora_Inicio_Turno, d.Hora_Fin_Turno, d.Horas_Bloqueadas,
           d.Acepta_Virtualidad, d.Hora_Inicio_Virtual, d.Hora_Fin_Virtual,
           GROUP_CONCAT(a.ID_Area) as Areas
    FROM docentes_activos d
    LEFT JOIN docente_area da ON d.ID_Docente = da.ID_Docente
    LEFT JOIN areas_academicas a ON da.ID_Area = a.ID_Area
    WHERE d.Disponible = 1
    GROUP BY d.ID_Docente
""", engine)

docentes = df_docentes.to_dict('records')
for d in docentes:
    d['Areas'] = [int(x) for x in str(d['Areas']).split(',')] if pd.notna(d['Areas']) else []

# 2. Extracción de Espacios (Con el nombre correcto Capacidad_Maxima)
df_espacios = pd.read_sql("SELECT ID_Espacio, Nombre_Espacio, Capacidad_Maxima as Capacidad, Tipo_Espacio FROM espacios_fisicos", engine)
espacios = df_espacios.to_dict('records')

# ==========================================
# 🛑 FUNCIÓN TRADUCTORA A PRUEBA DE FALLOS (Fix "0 days 08")
# ==========================================
def extraer_hora(valor_tiempo):
    if pd.isna(valor_tiempo): return None
    val_str = str(valor_tiempo).strip()
    if val_str == '00:00:00' or val_str == '0': return None
    # Si Pandas le agregó "days" (Ej: '0 days 08:00:00')
    if 'days' in val_str:
        return int(val_str.split('days')[1].split(':')[0].strip())
    # Si es un string normal (Ej: '08:00:00')
    return int(val_str.split(':')[0])

# ==========================================
# 🛑 REGLA INTELIGENTE: MODALIDAD VIRTUAL VS PRESENCIAL
# ==========================================
def docente_puede_dar_clase(docente, hora_int):
    # Filtro de bloqueos
    if pd.notna(docente['Horas_Bloqueadas']) and str(docente['Horas_Bloqueadas']).strip() != "":
        bloqueadas = [int(h.strip()) for h in str(docente['Horas_Bloqueadas']).split(',') if h.strip().isdigit()]
        if hora_int in bloqueadas: return False 
            
    es_horario_virtual = hora_int >= 17 
    
    if es_horario_virtual:
        if docente.get('Acepta_Virtualidad', 0) == 0: return False 
        ini_v = extraer_hora(docente['Hora_Inicio_Virtual']) or 17
        fin_v = extraer_hora(docente['Hora_Fin_Virtual']) or 21
        return ini_v <= hora_int < fin_v
    else:
        ini_p = extraer_hora(docente['Hora_Inicio_Turno'])
        fin_p = extraer_hora(docente['Hora_Fin_Turno'])
        # Si el resultado es None, significa que solo da clases virtuales (00:00:00)
        if ini_p is None or fin_p is None: return False
        return ini_p <= hora_int < fin_p

print(f"✅ Datos listos: {len(docentes)} docentes DISPONIBLES y {len(espacios)} aulas físicas.")

Cargando logística desde la BD...
✅ Datos listos: 4 docentes DISPONIBLES y 8 aulas físicas.


In [12]:
# ==========================================
# CELDA 2: Ingesta de la Demanda Proyectada
# ==========================================
print("📥 Cargando demanda proyectada (IPAC-2026)...")
df_demanda = pd.read_csv('../data/demanda_proyectada_2026.csv')

clases = []
for _, row in df_demanda.iterrows():
    clases.append({
        'ID_Clase': int(row['ID_Clase']),
        'Codigo_Oficial': row['Codigo_Oficial'],
        'Nombre_Clase': row['Nombre_Clase'],
        'ID_Area': int(row['ID_Area']),
        'Cupos_Estimados': int(row['Cupos_Estimados']),
        'Hora_Sugerida': str(row['Hora_Sugerida']).strip()
    })

print(f"✅ Se han cargado {len(clases)} asignaturas con demanda confirmada.")

📥 Cargando demanda proyectada (IPAC-2026)...
✅ Se han cargado 20 asignaturas con demanda confirmada.


In [13]:
# ==========================================
# CELDA 3: Modelo de Restricciones (CP-SAT Flexible con Variables de Holgura)
# ==========================================
model = cp_model.CpModel()
asignaciones = {}
slack_vars = {} # 🛑 LA MAGIA DE LA HOLGURA (Para evitar colapsos)
horas_disponibles = list(range(7, 21)) 

print("⚙️ Generando malla de variables cruzadas (Modo 100% Flexible)...")

# 1. VARIABLES: Permitimos TODAS las horas válidas para el docente
for c in clases:
    c_id = c['ID_Clase']
    for h in horas_disponibles:
        for d in docentes:
            if c['ID_Area'] in d['Areas'] and docente_puede_dar_clase(d, h):
                for r in espacios:
                    var_name = f'A_c{c_id}_d{d["ID_Docente"]}_r{r["ID_Espacio"]}_h{h}'
                    asignaciones[(c_id, d['ID_Docente'], r['ID_Espacio'], h)] = model.NewBoolVar(var_name)

# 2. RESTRICCIONES DURAS (Leyes físicas de tiempo y espacio)
for d in docentes:
    for h in horas_disponibles:
        model.AddAtMostOne([asignaciones[k] for k in asignaciones if k[1] == d['ID_Docente'] and k[3] == h])

for r in espacios:
    for h in horas_disponibles:
        model.AddAtMostOne([asignaciones[k] for k in asignaciones if k[2] == r['ID_Espacio'] and k[3] == h])

costos = []

# 3. RESTRICCIÓN FLEXIBLE DE DEMANDA (El Salvador del Modelo)
for c in clases:
    c_id = c['ID_Clase']
    vars_clase = [asignaciones[k] for k in asignaciones if k[0] == c_id]
    
    if vars_clase:
        # Creamos una variable para los estudiantes que se quedarán sin cupo por falta de recursos
        slack = model.NewIntVar(0, int(c['Cupos_Estimados']), f'slack_{c_id}')
        slack_vars[c_id] = slack
        
        # Fórmula: (Capacidad asignada en aulas) + (Alumnos sin cupo) >= Demanda Estimada
        model.Add(sum(
            int(float(next(esp['Capacidad'] for esp in espacios if esp['ID_Espacio'] == k[2])) * 1.2) * asignaciones[k]
            for k in asignaciones if k[0] == c_id
        ) + slack >= c['Cupos_Estimados'])
        
        # Penalizamos gigantescamente a la IA por usar la holgura (Costo de 5000)
        # Esto la obliga a intentar acomodar a todos antes de rendirse.
        costos.append(slack * 5000)

# 4. FUNCIÓN OBJETIVO: Costos por Sección y Preferencias del Censo
for k, var in asignaciones.items():
    c_id, d_id, r_id, h = k
    c_info = next(c for c in clases if c['ID_Clase'] == c_id)
    
    hora_ideal = -1
    if c_info['Hora_Sugerida'] != 'Sin preferencia':
        try:
            hora_ideal = int(str(c_info['Hora_Sugerida']).split(':')[0])
        except:
            pass
            
    # Costo base por abrir una sección (le duele 100 puntos)
    costo_seccion = 100 
    
    # RECOMPENSA: Si el algoritmo acierta la hora del Censo, se le da un descuento (le duele solo 10 puntos)
    if h == hora_ideal:
        costo_seccion = 10 
        
    costos.append(costo_seccion * var)

# Minimizar la suma de todos los costos
model.Minimize(sum(costos))

print("✅ Ecuaciones generadas con Priorización Flexible y Holgura. Listo para resolver.")

⚙️ Generando malla de variables cruzadas (Modo 100% Flexible)...
✅ Ecuaciones generadas con Priorización Flexible y Holgura. Listo para resolver.


In [14]:
# ==========================================
# CELDA 4: Ejecución del Solucionador y Reporte (Cupos Inteligentes)
# ==========================================
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 120.0
print("🚀 Iniciando Motor de Optimización Google OR-Tools...\n")

status = solver.Solve(model)
horarios_generados = []
alertas = 0

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print("✅ ¡BINGO! Se encontró una distribución logística óptima.\n")
    
    for k, variable in asignaciones.items():
        if solver.Value(variable) == 1:
            c_id, d_id, r_id, h = k
            c_info = next(c for c in clases if c['ID_Clase'] == c_id)
            d_info = next(d for d in docentes if d['ID_Docente'] == d_id)
            r_info = next(r for r in espacios if r['ID_Espacio'] == r_id)
            
            modalidad = "🌐 VIRTUAL" if h >= 17 else "🏫 PRESENCIAL"
            
            # 🛑 LA NUEVA MAGIA: CÁLCULO DE CUPOS INTELIGENTES
            demanda_real = c_info['Cupos_Estimados']
            capacidad_fisica_aula = int(float(r_info['Capacidad']))
            
            if demanda_real <= capacidad_fisica_aula:
                # Si la demanda es pequeña, habilitamos la demanda exacta + 5 cupos de margen de seguridad
                cupos_a_habilitar = demanda_real + 5
            else:
                # Si la demanda desborda el aula, habilitamos la capacidad máxima del aula + 20% de sobrecupo
                cupos_a_habilitar = int(capacidad_fisica_aula * 1.2)
            
            horarios_generados.append({
                'ID_Clase': c_id, 'Codigo': c_info['Codigo_Oficial'], 'Asignatura': c_info['Nombre_Clase'],
                'ID_Docente': d_id, 'Docente': d_info['Nombre'],
                'ID_Espacio': r_id, 'Aula': r_info['Nombre_Espacio'],
                'Hora_Inicio': f"{h:02d}:00:00", 'Hora_Fin': f"{(h+1):02d}:00:00",
                'Modalidad': modalidad, 'Cupos_Habilitados': cupos_a_habilitar,
                'Demanda_Real': demanda_real # Lo agregamos para que lo veas visualmente
            })
            
    df_horarios = pd.DataFrame(horarios_generados)
    
    # Análisis de déficit
    df_malla = pd.read_sql("SELECT ID_Clase, Codigo_Oficial, Nombre_Clase, ID_Area FROM Malla_Curricular", engine)
    
    for c in clases:
        id_c = c['ID_Clase']
        cupos_necesarios = c['Cupos_Estimados']
        # Calculamos la cobertura en base a la capacidad máxima que prestan las aulas asignadas
        cupos_logrados = sum([int(float(next(esp['Capacidad'] for esp in espacios if esp['ID_Espacio'] == k[2])) * 1.2) 
                              for k in asignaciones if k[0] == id_c and solver.Value(asignaciones[k]) == 1])
        
        if cupos_logrados < cupos_necesarios:
            alertas += 1
            codigo_oficial = df_malla[df_malla['ID_Clase'] == id_c]['Codigo_Oficial'].values[0]
            clase_nombre = df_malla[df_malla['ID_Clase'] == id_c]['Nombre_Clase'].values[0]
            area_id = int(df_malla[df_malla['ID_Clase'] == id_c]['ID_Area'].values[0])
            area_nombre = pd.read_sql(f"SELECT Nombre_Area FROM areas_academicas WHERE ID_Area = {area_id}", engine).values[0][0]
            
            print(f"⚠️ DÉFICIT EN: {codigo_oficial} - {clase_nombre}")
            print(f"   - Demanda Proyectada: {cupos_necesarios} alumnos.")
            print(f"   - Cobertura Máxima Posible: {cupos_logrados} alumnos.")
            print(f"   - 💡 SUGERENCIA: Quedaron {int(cupos_necesarios - cupos_logrados)} alumnos sin cupo físico. Requiere abrir más secciones con un Ing. en '{area_nombre}'.")
            print("-" * 60)
            
    if alertas == 0:
        print("✅ EXCELENTE: Usando el margen de tolerancia, se cubrió el 100% de la demanda proyectada sin desperdiciar recursos.\n")
        
    def color_modalidad(val):
        return 'color: #00BFFF; font-weight: bold' if 'VIRTUAL' in str(val) else 'color: #32CD32'
        
    display(df_horarios[['Codigo', 'Asignatura', 'Docente', 'Aula', 'Hora_Inicio', 'Hora_Fin', 'Modalidad', 'Demanda_Real', 'Cupos_Habilitados']]\
            .sort_values(['Hora_Inicio', 'Codigo'])\
            .style.applymap(color_modalidad, subset=['Modalidad']))
else:
    print("❌ CAOS LOGÍSTICO: El sistema no encontró solución. Hay demasiada demanda y muy pocos recursos (aulas/profesores).")

🚀 Iniciando Motor de Optimización Google OR-Tools...

✅ ¡BINGO! Se encontró una distribución logística óptima.

⚠️ DÉFICIT EN: IS-802 - Ingeniería del Software
   - Demanda Proyectada: 3 alumnos.
   - Cobertura Máxima Posible: 0 alumnos.
   - 💡 SUGERENCIA: Quedaron 3 alumnos sin cupo físico. Requiere abrir más secciones con un Ing. en 'Industria del Software'.
------------------------------------------------------------
⚠️ DÉFICIT EN: ISC-204 - Paradigmas de Programación
   - Demanda Proyectada: 3 alumnos.
   - Cobertura Máxima Posible: 0 alumnos.
   - 💡 SUGERENCIA: Quedaron 3 alumnos sin cupo físico. Requiere abrir más secciones con un Ing. en 'Ingeniería de Software'.
------------------------------------------------------------


,Codigo,Asignatura,Docente,Aula,Hora_Inicio,Hora_Fin,Modalidad,Demanda_Real,Cupos_Habilitados
0,IS-711,Diseño Digital,Ing. Edis Francisco Romero,Aula 401,08:00:00,09:00:00,🏫 PRESENCIAL,5,10
19,ISC-102,Programación Estructurada,Ing. Oscar Guillermo Hernández,Aula 321,08:00:00,09:00:00,🏫 PRESENCIAL,7,12
2,IS-702,Análisis y Diseño de Sistemas,Ing. Edis Francisco Romero,Aula 321,09:00:00,10:00:00,🏫 PRESENCIAL,7,12
1,IS-811,Seguridad Informática,Ing. Elias Emilio Flores,Aula 323,09:00:00,10:00:00,🏫 PRESENCIAL,5,10
9,IS-906,Topicos Especiales y Avanzados,Ing. Oscar Guillermo Hernández,Aula 322,09:00:00,10:00:00,🏫 PRESENCIAL,7,7
16,IS-115,Seminario de Investigación,Ing. Oscar Guillermo Hernández,Aula 402,10:00:00,11:00:00,🏫 PRESENCIAL,8,13
10,IS-910,Teoría de la Simulación,Ing. Asalia Alejandra Zavala,Aula 323,10:00:00,11:00:00,🏫 PRESENCIAL,8,13
17,ISC-101,Introducción a la Ingeniería en Sistemas Computacionales,Ing. Elias Emilio Flores,Aula 322,10:00:00,11:00:00,🏫 PRESENCIAL,19,7
4,IS-903,Auditoría Informática,Ing. Elias Emilio Flores,Aula 401,11:00:00,12:00:00,🏫 PRESENCIAL,7,7
14,IS-913,Diseño de Compiladores,Ing. Asalia Alejandra Zavala,Aula 402,11:00:00,12:00:00,🏫 PRESENCIAL,9,9


In [15]:
# ==========================================
# CELDA 5: Volcado a MySQL (Oferta Final)
# ==========================================
if horarios_generados:
    try:
        with engine.begin() as conn:
            conn.execute(text("TRUNCATE TABLE oferta_academica_generada"))
            for h in horarios_generados:
                dias_asig = "Lu,Ma,Mi,Ju" 
                query = text("""
                    INSERT INTO oferta_academica_generada 
                    (Periodo_Academico, ID_Clase, ID_Docente, ID_Espacio, Dias, Hora_Inicio, Hora_Fin, Cupos_Maximos, Aprobado_Por_Jefatura)
                    VALUES (:per, :clase, :doc, :esp, :dias, :h_ini, :h_fin, :cupos, 0)
                """)
                conn.execute(query, {
                    'per': '1-2026', 
                    'clase': h['ID_Clase'], 'doc': h['ID_Docente'], 'esp': h['ID_Espacio'],
                    'dias': dias_asig, 'h_ini': h['Hora_Inicio'], 'h_fin': h['Hora_Fin'], 
                    'cupos': h['Cupos_Habilitados'] # Usamos el cupo inteligente
                })
        print("💾 ¡ÉXITO! La Oferta Académica Oficial ha sido guardada en la base de datos (Tabla: oferta_academica_generada).")
    except Exception as e:
        print(f"❌ Error crítico al escribir en MySQL: {e}")

💾 ¡ÉXITO! La Oferta Académica Oficial ha sido guardada en la base de datos (Tabla: oferta_academica_generada).
